# Lab01 - Mật mã học cổ điển
**Nhóm 16 - NT101.R11.2**  
**Hà Nguyễn Hiếu An (24520043), Nguyễn Hữu Nghĩa (24521144)**

# Nhiệm vụ 2.1 - Caesar
## Quy ước
Biểu diễn `A= 0, ..., Z= 25`:
- Mã hóa: $C = (p + k) \bmod 26$.
- Giải mã: $p = (C - k) \bmod 26$.

## Tự động tìm khóa
Thử cả 26 khóa `0..25`, giải mã mỗi khóa và chấm điểm:
1. Thống kê tần suất `A–Z`, tính chi-square so với tần suất tiếng Anh. Giá trị nhỏ tốt hơn.
2. Trừ phần thưởng cho từ tiếng Anh phổ biến.
3. Chọn một ứng viên có điểm thấp nhất, in ra C và p.

In [ ]:
import re
from collections import Counter

#Nguồn: https://en.wikipedia.org/wiki/Letter_frequency
CAESAR_ENGLISH_FREQUENCIES = (
    8.167, 1.492, 2.782, 4.253, 12.702, 2.228, 2.015, 6.094, 6.966,
    0.153, 0.772, 4.025, 2.406, 6.749, 7.507, 1.929, 0.095, 5.987,
    6.327, 9.056, 2.758, 0.978, 2.360, 0.150, 1.974, 0.074,
)
CAESAR_COMMON_WORDS = frozenset(
    "the of and to in is it that for on with as was are be this by from "
    "at or an a not have has but they you we can will their which".split()
)


def caesar_encrypt(text, key):
    """Dịch"""
    if not isinstance(key, int) or isinstance(key, bool):
        raise ValueError("Phải là số nguyên")
    shift = key % 26
    result = []
    for character in text:
        if "A" <= character <= "Z":
            result.append(chr((ord(character) - ord("A") + shift) % 26 + ord("A")))
        elif "a" <= character <= "z":
            result.append(chr((ord(character) - ord("a") + shift) % 26 + ord("a")))
        else:
            result.append(character)
    return "".join(result)


def caesar_decrypt(ciphertext, key):
    if not isinstance(key, int) or isinstance(key, bool):
        raise ValueError("Khóa Caesar phải là số nguyên.")
    return caesar_encrypt(ciphertext, -key)


def caesar_english_score(text):
    letters = re.findall(r"[a-z]", text.lower())
    if not letters:
        raise ValueError("Cần có chữ cái")
    counts = Counter(letters)
    total = len(letters)
    chi_square = 0.0
    for index, frequency in enumerate(CAESAR_ENGLISH_FREQUENCIES):
        expected = total * frequency / 100
        observed = counts[chr(ord("a") + index)]
        chi_square += (observed - expected) ** 2 / expected
    words = re.findall(r"[a-z]+", text.lower())
    common_count = sum(word in CAESAR_COMMON_WORDS for word in words)
    return chi_square - 4.0 * common_count


def caesar_brute_force(ciphertext):
    """Trả về: k, p, score."""
    candidates = []
    for key in range(26):
        plaintext = caesar_decrypt(ciphertext, key)
        candidates.append({
            "key": key,
            "plaintext": plaintext,
            "score": caesar_english_score(plaintext),
        })
    return min(candidates, key=lambda candidate: candidate["score"])

### Giải ciphertext 
Nguồn: https://pastecode.io/s/y3vj3cqe

In [ ]:
CAESAR_TASK_CIPHERTEXT = 'Max NBM bl t extwbgz bglmbmnmbhg ngwxk OGN-AVF, lixvbtebsbgz bg max ybxew hy bgyhkftmbhg mxvaghehzr. Xlmtueblaxw pbma t fbllbhg mh yhlmxk bgghotmbhg tgw xqvxeexgvx bg BM xwnvtmbhg tgw kxlxtkva, NBM hyyxkl t pbwx ktgzx hy ngwxkzktwntmx tgw ihlmzktwntmx ikhzktfl tbfxw tm ikhwnvbgz abzaer ldbeexw ikhyxllbhgtel. Max ngboxklbmr bl kxvhzgbsxw yhk bml vnmmbgz-xwzx kxlxtkva bg tkxtl ebdx vruxklxvnkbmr, tkmbybvbte bgmxeebzxgvx, tgw lhymptkx xgzbgxxkbgz. Pbma lmtmx-hy-max-tkm ytvbebmbxl tgw t lmkhgz xfiatlbl hg vheetuhktmbhg pbma bgwnlmkr, NBM xjnbil lmnwxgml pbma uhma maxhkxmbvte dghpexwzx tgw iktvmbvte ldbeel mh makbox bg max ktibwer xoheobgz mxva bgwnlmkr.'

caesar_task_result = caesar_brute_force(CAESAR_TASK_CIPHERTEXT)
print("k = ", caesar_task_result["key"])
print("score (nhỏ hơn tốt hơn):", round(caesar_task_result["score"], 3))
print("p = \n")
print(caesar_task_result["plaintext"])

k =  19
Điểm thống kê (nhỏ hơn tốt hơn): -84.065
p = 

The UIT is a leading institution under VNU-HCM, specializing in the field of information technology. Established with a mission to foster innovation and excellence in IT education and research, UIT offers a wide range of undergraduate and postgraduate programs aimed at producing highly skilled professionals. The university is recognized for its cutting-edge research in areas like cybersecurity, artificial intelligence, and software engineering. With state-of-the-art facilities and a strong emphasis on collaboration with industry, UIT equips students with both theoretical knowledge and practical skills to thrive in the rapidly evolving tech industry.


### Nhập dữ liệu Caesar

In [ ]:
CAESAR_INTERACTIVE = True

if CAESAR_INTERACTIVE:
    try:
        caesar_mode = input("Caesar: 1 = mã hóa, 2 = giải mã, 3 = tìm khóa: ").strip()
        if caesar_mode not in {"1", "2", "3"}:
            raise ValueError("Chế độ phải là 1, 2 hoặc 3.")
        caesar_input_text = input("Nhập văn bản: ")
        if caesar_mode == "3":
            caesar_input_result = caesar_brute_force(caesar_input_text)
            print("Khóa mã hóa:", caesar_input_result["key"])
            print("Bản rõ:", caesar_input_result["plaintext"])
        else:
            caesar_input_key = int(input("Nhập khóa nguyên: "))
            caesar_transform = caesar_encrypt if caesar_mode == "1" else caesar_decrypt
            print("Kết quả:", caesar_transform(caesar_input_text, caesar_input_key))
    except ValueError as error:
        print("Lỗi:", error)
else:
    print("Đổi CAESAR_INTERACTIVE = True để sử dụng.")

Nhập liệu Caesar đang tắt. Đổi CAESAR_INTERACTIVE = False để sử dụng.


## Kiểm thử Caesar
Kiểm tra ví dụ chuẩn, quay vòng, dấu câu/Unicode, khóa âm/lớn, các khóa `0..25`, đầu vào không hợp lệ và kết quả đề.
`unittest` vẫn kiểm tra khi Python bật tối ưu hóa (không phụ thuộc `assert`).

In [4]:
import unittest


class TestCaesar(unittest.TestCase):
    def test_known_example(self):
        self.assertEqual(caesar_encrypt("Hello, World!", 3), "Khoor, Zruog!")
        self.assertEqual(caesar_decrypt("Khoor, Zruog!", 3), "Hello, World!")

    def test_wrap_and_preservation(self):
        self.assertEqual(caesar_encrypt("Zz Aa! 123 à Đ", 1), "Aa Bb! 123 à Đ")
        self.assertEqual(caesar_encrypt("", 3), "")

    def test_keys_and_round_trip(self):
        self.assertEqual(caesar_encrypt("Abc", -1), "Zab")
        self.assertEqual(caesar_encrypt("Abc", 29), "Def")
        for key in range(-26, 53):
            with self.subTest(key=key):
                original = "Attack at dawn! Zz, à 2026."
                self.assertEqual(caesar_decrypt(caesar_encrypt(original, key), key), original)

    def test_invalid_inputs(self):
        for key in ("3", 1.5, True):
            with self.subTest(key=key):
                with self.assertRaises(ValueError):
                    caesar_encrypt("abc", key)
                with self.assertRaises(ValueError):
                    caesar_decrypt("abc", key)
        for text in ("", "123 ! à"):
            with self.assertRaises(ValueError):
                caesar_brute_force(text)

    def test_brute_force_all_keys(self):
        sample = "The university is a leading institution in the field of information technology."
        for key in range(26):
            with self.subTest(key=key):
                result = caesar_brute_force(caesar_encrypt(sample, key))
                self.assertEqual(result["key"], key)
                self.assertEqual(result["plaintext"], sample)

    def test_task_ciphertext(self):
        result = caesar_brute_force(CAESAR_TASK_CIPHERTEXT)
        self.assertEqual(result["key"], 19)
        self.assertTrue(result["plaintext"].startswith("The UIT is a leading institution"))
        self.assertEqual(caesar_encrypt(result["plaintext"], result["key"]), CAESAR_TASK_CIPHERTEXT)


caesar_test_result = unittest.TextTestRunner(verbosity=2).run(
    unittest.defaultTestLoader.loadTestsFromTestCase(TestCaesar)
)
if not caesar_test_result.wasSuccessful():
    raise RuntimeError("Kiểm thử Caesar thất bại.")

test_brute_force_all_keys (__main__.TestCaesar.test_brute_force_all_keys) ... 

ok


test_invalid_inputs (__main__.TestCaesar.test_invalid_inputs) ... 

ok


test_keys_and_round_trip (__main__.TestCaesar.test_keys_and_round_trip) ... 

...

test_task_ciphertext (__main__.TestCaesar.test_task_ciphertext) ... 

ok


test_wrap_and_preservation (__main__.TestCaesar.test_wrap_and_preservation) ... 

ok


----------------------------------------------------------------------
Ran 6 tests in 0.052s

OK


## Tài liệu tham khảo
- Ciphertext đề: https://pastecode.io/s/y3vj3cqe.
- Tham khảo thuật toán/thống kê: https://en.wikipedia.org/wiki/Letter_frequency.